# Configuração Automatizada da Integração Jira no E-Stack

Este notebook automatiza as etapas de configuração necessárias no backend do E-Stack para integrar um projeto com o Jira:
1. **Usuário de Sistema:** Garante a existência e ativação do usuário `e-stack@elumini-it.com`.
2. **Seleção do Projeto:** Localiza o projeto no E-Stack e suas respectivas colunas do Kanban.
3. **Validação da API do Jira:** Testa a conectividade, credenciais e busca metadados (Project ID e Status) no Jira.
4. **Persistência de Dados:** Salva/atualiza `JiraIntegrationConfig` e mapeamentos `JiraIntegrationColumnsMapping` no PostgreSQL.
5. **Instruções de Webhook:** Gera a URL e parâmetros para registro no painel de administração do Jira.

In [ ]:
import os
import sys
import uuid
from dotenv import load_dotenv, find_dotenv

# Carrega variáveis de ambiente do .env
load_dotenv(find_dotenv())

# Adiciona o diretório raiz do backend ao sys.path
backend_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if backend_path not in sys.path:
    sys.path.append(backend_path)

from src.database.database import create_session
from src.database import models
import httpx
import nest_asyncio

nest_asyncio.apply()
print("Ambiente e dependências inicializados com sucesso.")

## 1. Parâmetros da Integração

Preencha as variáveis abaixo com as credenciais do Jira e informações do projeto no E-Stack.

In [ ]:
# --- Configurações do Projeto no E-Stack ---
# Defina o UUID do projeto ou deixe como None/vazio para listar os projetos disponíveis
PROJECT_ID = "<ID DO PROJETO>"

# --- Credenciais e Parâmetros do Jira ---
JIRA_BASE_URL = "https://<SUA EMPRESA>.atlassian.net"  # URL base sem barra final
JIRA_EMAIL = "<SUA EMAIL>"
JIRA_API_TOKEN = "<TOKEN>"
JIRA_PROJECT_KEY = ""
JIRA_PROJECT_ID = ""  # Opcional: se vazio, será buscado automaticamente via API do Jira
DEFAULT_ISSUE_TYPE = "Tarefa"

# --- Mapeamento: Status do Jira -> Nome da Coluna Kanban no E-Stack ---
# Exemplo de mapeamento:
STATUS_MAPPING = {
    "Itens Pendentes": "Itens Pendentes",
    "Em Andamento": "Em Andamento",
    "Itens concluídos": "Itens concluídos"
}

print("Variáveis de ambiente e dependências inicializadas com sucesso.")

## 2. Garantir Usuário de Sistema (`e-stack@elumini-it.com`)

O backend exige que o usuário de sistema exista para associar as histórias criadas a partir do Jira.

In [ ]:
SYSTEM_USER_EMAIL = "e-stack@elumini-it.com"
SYSTEM_USER_NAME = "E-Stack"

with create_session() as session:
    user = session.query(models.User).filter(models.User.email == SYSTEM_USER_EMAIL).first()
    if not user:
        user = models.User(
            id=str(uuid.uuid4()),
            email=SYSTEM_USER_EMAIL,
            is_active=True,
            is_admin=False
        )
        session.add(user)
        session.commit()
        print(f"✓ Usuário de sistema criado com sucesso: {user.email} (ID: {user.id})")
    else:
        if not user.is_active:
            user.is_active = True
            session.commit()
            print(f"✓ Usuário de sistema reativado: {user.email} (ID: {user.id})")
        else:
            print(f"✓ Usuário de sistema já existe e está ativo: {user.email} (ID: {user.id})")

## 3. Seleção do Projeto e Colunas do Kanban

Verifica a existência do projeto e carrega as colunas configuradas na tabela `project_kanban_column`.

In [ ]:
with create_session() as session:
    if not PROJECT_ID or PROJECT_ID.startswith("<"):
        print("ℹ PROJECT_ID não informado. Listando projetos ativos disponíveis:")
        active_projects = session.query(models.Project).filter(models.Project.is_active == True).all()
        for p in active_projects:
            print(f"  - ID: {p.id} | Nome: {p.name}")
        raise ValueError("Por favor, configure a variável PROJECT_ID na célula de parâmetros antes de continuar.")

    project = session.query(models.Project).filter(models.Project.id == PROJECT_ID).first()
    if not project:
        raise ValueError(f"Projeto com ID '{PROJECT_ID}' não encontrado no banco de dados.")

    print(f"✓ Projeto encontrado: {project.name} (ID: {project.id})")
    
    columns = (
        session.query(models.ProjectKanbanColumn)
        .filter(models.ProjectKanbanColumn.project_id == project.id)
        .order_by(models.ProjectKanbanColumn.position)
        .all()
    )
    
    if not columns:
        raise ValueError(f"Nenhuma coluna de Kanban encontrada para o projeto {project.name}.")
    
    column_by_name = {}
    print(f"✓ Colunas Kanban disponíveis ({len(columns)} colunas):")
    for col in columns:
        column_by_name[col.name.strip().lower()] = col
        print(f"  - Posição {col.position}: '{col.name}' (ID: {col.id})")

## 4. Teste de Conexão com o Jira e Busca de Metadados

Valida as credenciais na API REST v3 do Jira, recupera o `jira_project_id` e a lista de status de transição do projeto.

In [ ]:
clean_base_url = JIRA_BASE_URL.strip().rstrip("/")
auth = (JIRA_EMAIL.strip(), JIRA_API_TOKEN.strip())

jira_statuses = {}
with httpx.Client(base_url=clean_base_url, auth=auth, timeout=15.0) as client:
    # 1. Autenticação
    resp_user = client.get("/rest/api/3/myself")
    if resp_user.status_code != 200:
        raise RuntimeError(f"Falha na autenticação com Jira ({clean_base_url}): HTTP {resp_user.status_code} - {resp_user.text}")
    user_data = resp_user.json()
    print(f"✓ Conectado ao Jira com sucesso! Usuário: {user_data.get('displayName')} ({user_data.get('emailAddress')})")

    # 2. Dados do Projeto
    resp_project = client.get(f"/rest/api/3/project/{JIRA_PROJECT_KEY.strip()}")
    if resp_project.status_code != 200:
        raise RuntimeError(f"Projeto '{JIRA_PROJECT_KEY}' não encontrado no Jira: HTTP {resp_project.status_code} - {resp_project.text}")
    
    project_data = resp_project.json()
    resolved_jira_project_id = str(project_data.get("id"))
    print(f"✓ Projeto Jira validado: '{project_data.get('name')}' (Chave: {JIRA_PROJECT_KEY}, ID: {resolved_jira_project_id})")

    if not JIRA_PROJECT_ID:
        JIRA_PROJECT_ID = resolved_jira_project_id

    # 3. Status disponíveis no Jira
    resp_statuses = client.get(f"/rest/api/3/project/{JIRA_PROJECT_KEY.strip()}/statuses")
    if resp_statuses.status_code == 200:
        for it in resp_statuses.json():
            for st in it.get("statuses", []):
                st_name = st.get("name")
                st_id = str(st.get("id"))
                jira_statuses[st_name.strip().lower()] = {"id": st_id, "name": st_name}
                jira_statuses[st_id] = {"id": st_id, "name": st_name}
        
        unique_statuses = {s["id"]: s["name"] for s in jira_statuses.values()}
        print(f"✓ Status de transição identificados no Jira ({len(unique_statuses)}): {unique_statuses}")

## 5. Salvar Configurações no Banco de Dados

Grava os registros em `jira_integration_config` e `jira_integration_columns_mapping`.

In [ ]:
with create_session() as session:
    # 1. Cria ou atualiza JiraIntegrationConfig
    config = (
        session.query(models.JiraIntegrationConfig)
        .filter(models.JiraIntegrationConfig.project_id == PROJECT_ID)
        .first()
    )
    
    if not config:
        config = models.JiraIntegrationConfig(
            id=str(uuid.uuid4()),
            project_id=PROJECT_ID,
            jira_base_url=clean_base_url,
            jira_email=JIRA_EMAIL.strip(),
            jira_api_token=JIRA_API_TOKEN.strip(),
            jira_project_id=str(JIRA_PROJECT_ID),
            jira_project_key=JIRA_PROJECT_KEY.strip(),
            default_issue_type=DEFAULT_ISSUE_TYPE.strip()
        )
        session.add(config)
        session.flush()
        print(f"✓ Configuração principal inserida (Config ID: {config.id})")
    else:
        config.jira_base_url = clean_base_url
        config.jira_email = JIRA_EMAIL.strip()
        config.jira_api_token = JIRA_API_TOKEN.strip()
        config.jira_project_id = str(JIRA_PROJECT_ID)
        config.jira_project_key = JIRA_PROJECT_KEY.strip()
        config.default_issue_type = DEFAULT_ISSUE_TYPE.strip()
        session.flush()
        print(f"✓ Configuração principal atualizada (Config ID: {config.id})")

    # 2. Recria os mapeamentos de coluna
    (
        session.query(models.JiraIntegrationColumnsMapping)
        .filter(models.JiraIntegrationColumnsMapping.jira_integration_config_id == config.id)
        .delete()
    )
    
    mappings_count = 0
    for jira_status_ref, kanban_col_name in STATUS_MAPPING.items():
        kanban_col = column_by_name.get(kanban_col_name.strip().lower())
        if not kanban_col:
            print(f"  ⚠ Aviso: Coluna Kanban '{kanban_col_name}' não encontrada no projeto. Pulando...")
            continue
        
        status_info = jira_statuses.get(str(jira_status_ref).strip().lower())
        if status_info:
            status_id = status_info["id"]
            status_name = status_info["name"]
        else:
            status_id = str(jira_status_ref)
            status_name = str(jira_status_ref)
        
        mapping = models.JiraIntegrationColumnsMapping(
            id=str(uuid.uuid4()),
            jira_integration_config_id=str(config.id),
            project_kanban_column_id=kanban_col.id,
            jira_status_id=status_id,
            jira_status_name=status_name
        )
        session.add(mapping)
        mappings_count += 1
        print(f"✓ Mapeamento salvo: Status Jira '{status_name}' ({status_id}) -> Coluna Kanban '{kanban_col.name}' ({kanban_col.id})")

    session.commit()
    print(f"\n✓ Sucesso! {mappings_count} mapeamentos salvos no banco de dados.")

## 6. Configuração do Webhook no Jira

Execute a célula abaixo para visualizar os parâmetros exatos a serem cadastrados no Jira.

In [ ]:
backend_url = os.getenv("BACKEND_PUBLIC_URL", "https://estack-backend-1188d1c3e6ca.herokuapp.com").rstrip("/")
webhook_endpoint = f"{backend_url}/webhook/jira/"

print("=" * 75)
print(" CONFIGURAÇÃO NO BACKEND CONCLUÍDA!")
print("=" * 75)
print("\nSiga as instruções abaixo para registrar o WebHook no Jira:\n")
print("1. Acesse: Jira > Configurações (engrenagem) > Sistema > Avançado > WebHooks")
print("2. Clique em '+ Criar um WebHook' (+ Create a WebHook)")
print("3. Preencha os campos com os valores:")
print(f"   • Nome: E-Stack Integration - {project.name}")
print("   • Status: Habilitado (Enabled)")
print(f"   • URL: {webhook_endpoint}")
print(f"   • JQL Filter: project = \"{JIRA_PROJECT_KEY}\"")
print("   • Eventos de Issue: Marque [Criado], [Atualizado] e [Excluído]")
print("4. Clique em 'Salvar' (Save) no final da página.")
print("=" * 75)